# SPoRC Two-Stage Summarization Pipeline (thousands of episodes trial)

This notebook contains:
1. Dataset construction + pseudo-label generation
2. Extractor + Refiner model (Two-Stage)
3. Training loop (trial on first 250 episodes)
4. Evaluation on pseudo summaries

All logic is embedded in a single notebook for debugging & iteration.

In [1]:
import json
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk

from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModel, BartForConditionalGeneration
from collections import defaultdict
from rouge_score import rouge_scorer
from transformers import (
    AutoTokenizer,
    AutoModel,
    BartForConditionalGeneration
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


c:\Users\ddq\anaconda3\envs\pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
import json
import re
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util
import torch


class SPoRCDataset:
    BOILERPLATE_PATTERNS = [
        r"(?i)\bno guest\b.*?(?:\.\s*|$)",
        r"(?i)\bguest was not present\b.*?(?:\.\s*|$)",
        r"(?i)\bthere were no guests?\b.*?(?:\.\s*|$)",
        r"(?i)\bno guests?\b.*?(?:\.\s*|$)",
        r"(?i)\bno guest content\b.*?(?:\.\s*|$)",
        r"(?i)\bno guest was present in this transcript\b.*?(?:\.\s*|$)",
    ]

    def __init__(
        self,
        csv_path: str,
        summary_jsonl_path: str,
        bart_name: str = "facebook/bart-large",
        sent_emb_name: str = "all-mpnet-base-v2",
        device="cpu",
    ):
        self.df = pd.read_csv(csv_path)
        self.episodes = self._group_by_episode(self.df)

        self.tokenizer = AutoTokenizer.from_pretrained(bart_name)

        self.sent_embedder = SentenceTransformer(sent_emb_name).to(device)

        # 预先计算每个 episode 是否真的有 host/guest turn（避免训练“无guest模板”）
        self.role_present = self._build_role_presence()

        self.summaries = {}
        self._load_summaries_jsonl(summary_jsonl_path)

    @staticmethod
    def _safe_str(x) -> str:
        if x is None:
            return ""
        try:
            if pd.isna(x):
                return ""
        except Exception:
            pass
        return str(x)

    @classmethod
    def _strip_boilerplate(cls, s: str) -> str:
        if not isinstance(s, str):
            return ""
        out = s.strip()
        for pat in cls.BOILERPLATE_PATTERNS:
            out = re.sub(pat, "", out).strip()
        # 再把多余空白收一下
        out = re.sub(r"\s+", " ", out).strip()
        return out

    @staticmethod
    def _normalize_summary(x) -> str:
        if x is None:
            s = ""
        elif isinstance(x, str):
            s = x.strip()
        elif isinstance(x, dict):
            for k in ("summary", "text", "content", "output"):
                v = x.get(k)
                if isinstance(v, str):
                    s = v.strip()
                    break
            else:
                s = json.dumps(x, ensure_ascii=False).strip()
        elif isinstance(x, list):
            parts = [SPoRCDataset._normalize_summary(i) for i in x]
            s = " ".join([p for p in parts if p]).strip()
        else:
            s = str(x).strip()

        # 过滤常见占位符
        if s.upper() in {"N/A", "NA", "NULL", "NONE"}:
            return ""
        return s

    def _group_by_episode(self, df: pd.DataFrame):
        episodes = defaultdict(list)
        for _, row in df.iterrows():
            ep = self._safe_str(row.get("episode", ""))
            try:
                turn = int(row.get("turn", None))
            except Exception:
                continue

            episodes[ep].append(
                {
                    "turn": turn,
                    "role": self._safe_str(row.get("role", "")),
                    "speaker": self._safe_str(row.get("speaker", "")),
                    "text": self._safe_str(row.get("text", "")),
                }
            )

        for ep in episodes:
            episodes[ep] = sorted(episodes[ep], key=lambda x: x["turn"])
        return episodes

    def _build_role_presence(self):
        role_present = {}
        for ep, utts in self.episodes.items():
            has_host = any(u.get("role") == "host" and str(u.get("text","")).strip() != "" for u in utts)
            has_guest = any(u.get("role") == "guest" and str(u.get("text","")).strip() != "" for u in utts)
            role_present[ep] = {"host": has_host, "guest": has_guest}
        return role_present

    def _load_summaries_jsonl(self, path: str):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                ep = obj.get("episode", None)
                if ep is None:
                    continue
                ep = str(ep)

                g = self._normalize_summary(obj.get("global_summary"))
                h = self._normalize_summary(obj.get("host_summary"))
                ge = self._normalize_summary(obj.get("guest_summary"))

                # ✅ 去掉“无guest模板句”
                g = self._strip_boilerplate(g)
                h = self._strip_boilerplate(h)
                ge = self._strip_boilerplate(ge)

                # ✅ 如果这个 episode 根本没有 guest turn，就强制 guest_summary 置空（不训练、不评估）
                if ep in self.role_present and not self.role_present[ep]["guest"]:
                    ge = ""

                self.summaries[ep] = {"global": g, "host": h, "guest": ge}

    # ---------------- public API ----------------
    def episode_ids(self):
        return list(self.episodes.keys())

    def get_summary(self, episode: str, summary_type: str):
        if episode not in self.summaries:
            return ""
        return self.summaries[episode].get(summary_type, "")

    def encode_episode(self, episode: str, max_utt_len: int = 64):
        utts = self.episodes[episode]
        texts = [self._safe_str(u.get("text", "")) for u in utts]
        roles = [self._safe_str(u.get("role", "")) for u in utts]

        enc = self.tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_utt_len,
            return_tensors="pt",
        )
        return enc["input_ids"], enc["attention_mask"], roles, texts

    def build_pseudo_labels(self, episode: str, summary_type: str = "global", topk: int = 5):
        if episode not in self.summaries:
            return None

        summaries = self.summaries[episode]
        if summaries.get("global", "").strip() == "":
            return None

        summary_text = summaries.get(summary_type, "")
        if summary_text is None or summary_text.strip() == "":
            return None
        summary_text = summary_text.strip()

        utts = self.episodes.get(episode, [])
        if not utts:
            return None

        if summary_type == "host":
            idx_text = [(i, self._safe_str(u.get("text", "")))
                        for i, u in enumerate(utts) if self._safe_str(u.get("role","")) == "host"]
        elif summary_type == "guest":
            idx_text = [(i, self._safe_str(u.get("text", "")))
                        for i, u in enumerate(utts) if self._safe_str(u.get("role","")) == "guest"]
        else:
            idx_text = [(i, self._safe_str(u.get("text", ""))) for i, u in enumerate(utts)]

        idx_text = [(i, t) for i, t in idx_text if t.strip() != ""]
        if len(idx_text) == 0:
            return None

        indices, texts = zip(*idx_text)

        utt_emb = self.sent_embedder.encode(list(texts), convert_to_tensor=True)
        sum_emb = self.sent_embedder.encode(summary_text, convert_to_tensor=True)
        if sum_emb.dim() == 1:
            sum_emb = sum_emb.unsqueeze(0)

        sims = util.cos_sim(utt_emb, sum_emb).squeeze(1)
        k = min(topk, sims.size(0))
        topk_local = torch.topk(sims, k=k).indices.tolist()

        return [indices[i] for i in topk_local]


In [3]:
# =====cell 2: model_tds.py (embedded in notebook) =====

class UtteranceEncoder(nn.Module):
    """
    Encode each utterance into a fixed-size vector
    using a pretrained BART encoder.
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(bart_name).encoder
        self.encoder.to(device)
        
    def forward(self, input_ids, attention_mask):
        """
        input_ids:      [num_utts, seq_len]
        attention_mask: [num_utts, seq_len]

        returns:
            utt_emb: [num_utts, hidden_dim]
        """
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # 使用每句话的第一个 token（<s>）作为句向量
        utt_emb = outputs.last_hidden_state[:, 0]
        return utt_emb


class Extractor(nn.Module):
    """
    Simple extractor:
    Given utterance embeddings, predict an importance score
    for each utterance.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, utt_emb):
        """
        utt_emb: [num_utts, hidden_dim]

        returns:
            logits: [num_utts]
        """
        logits = self.classifier(utt_emb).squeeze(-1)
        return logits


class Refiner(nn.Module):
    """
    Abstractive summarizer based on BART.
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()
        self.model = BartForConditionalGeneration.from_pretrained(bart_name)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Standard BART forward.

        If labels is provided, returns training loss.
        """
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )


class TwoStageSummarizer(nn.Module):
    """
    Two-stage summarization model:
    1) UtteranceEncoder + Extractor (sentence selection)
    2) Refiner (BART generation)
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()

        self.encoder = UtteranceEncoder(bart_name)
        self.extractor = Extractor(hidden_dim=1024)  # bart-large hidden size
        self.refiner = Refiner(bart_name)

    def forward_extractor(self, input_ids, attention_mask):
        """
        Forward pass for extractor only.

        returns:
            logits:  [num_utts]
            utt_emb: [num_utts, hidden_dim]
        """
        utt_emb = self.encoder(input_ids, attention_mask)
        logits = self.extractor(utt_emb)
        return logits, utt_emb


In [4]:
# =====cell 3 =====
def train_extractor(
    model,
    dataset,
    episodes,
    summary_type="global",
    topk=5,
    lr=1e-4,
):
    model.encoder.eval()      # encoder frozen
    model.extractor.train()

    optimizer = torch.optim.AdamW(model.extractor.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()

    total_loss = 0.0
    count = 0

    for ep in episodes:
        pos_indices = dataset.build_pseudo_labels(
            ep, summary_type=summary_type, topk=topk
        )
        if pos_indices is None:
            continue

        input_ids, attn, _, _ = dataset.encode_episode(ep)
        input_ids = input_ids.to(device)
        attn = attn.to(device)

        with torch.no_grad():
            utt_emb = model.encoder(input_ids, attn)

        logits = model.extractor(utt_emb)
        labels = torch.zeros_like(logits)
        labels[pos_indices] = 1.0

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

    return total_loss / max(count, 1)


In [5]:
SUMMARY_PREFIX = {"global": "<GLOBAL>", "host": "<HOST>", "guest": "<GUEST>"}

def generate_summary(model, dataset, episode, summary_type="global", max_sentences=6, max_len=256):
    model.eval()

    input_ids, attn, roles, texts = dataset.encode_episode(episode)
    input_ids = input_ids.to(device)
    attn = attn.to(device)

    # ✅ 若该角色根本不存在，直接返回空（别让模型胡编“no guest”）
    if summary_type in ("host", "guest"):
        if not any(r == summary_type for r in roles):
            return ""

    with torch.no_grad():
        logits, _ = model.forward_extractor(input_ids, attn)

        if summary_type in ("host", "guest"):
            keep = torch.tensor([r == summary_type for r in roles], device=logits.device)
            logits = logits.masked_fill(~keep, -1e9)

        k = min(max_sentences, logits.size(0))
        topk = torch.topk(logits, k=k).indices

    selected = [texts[i] for i in sorted(topk.tolist()) if texts[i].strip() != ""]
    if len(selected) == 0:
        return ""

    tok = dataset.tokenizer
    concat_text = SUMMARY_PREFIX[summary_type] + " " + " ".join(selected)

    enc = tok(concat_text, truncation=True, max_length=max_len, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.refiner.model.generate(
            **enc,
            max_length=150,
            min_length=30,
            num_beams=4,
            no_repeat_ngram_size=3,
        )

    return tok.decode(out[0], skip_special_tokens=True).strip()


In [6]:
# =====cell 5: evaluation.py (embedded in notebook) =====

scorer = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)


def eval_episode(
    model,
    dataset,
    episode,
    summary_type="global",
):
    """
    Evaluate ONE episode using ROUGE-L.
    """
    ref = dataset.get_summary(episode, summary_type)
    if ref == "":
        return None

    pred = generate_summary(
        model, dataset, episode, summary_type=summary_type
    )

    score = scorer.score(ref, pred)
    return score["rougeL"].fmeasure


def evaluate(
    model,
    dataset,
    episodes,
    summary_type="global",
):
    """
    Evaluate multiple episodes.
    """
    scores = []
    for ep in episodes:
        s = eval_episode(model, dataset, ep, summary_type)
        if s is not None:
            scores.append(s)

    if len(scores) == 0:
        return None

    return sum(scores) / len(scores)


In [7]:
SUMMARY_PREFIX = {"global": "<GLOBAL>", "host": "<HOST>", "guest": "<GUEST>"}

def train_refiner_on_episode(model, dataset, episode, summary_type, optimizer, max_len=256):
    ref = dataset.get_summary(episode, summary_type)
    if ref is None or ref.strip() == "":
        return None

    input_ids, attn, roles, texts = dataset.encode_episode(episode)
    input_ids = input_ids.to(device)
    attn = attn.to(device)

    # ✅ 若该角色根本不存在，直接跳过训练
    if summary_type in ("host", "guest"):
        if not any(r == summary_type for r in roles):
            return None

    with torch.no_grad():
        logits, _ = model.forward_extractor(input_ids, attn)

        if summary_type in ("host", "guest"):
            keep = torch.tensor([r == summary_type for r in roles], device=logits.device)
            logits = logits.masked_fill(~keep, -1e9)

        topk = torch.topk(logits, k=min(6, logits.size(0))).indices

    selected = [texts[i] for i in sorted(topk.tolist()) if texts[i].strip() != ""]
    if len(selected) == 0:
        return None

    tok = dataset.tokenizer
    concat_text = SUMMARY_PREFIX[summary_type] + " " + " ".join(selected)

    enc = tok(concat_text, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    tgt = tok(ref, truncation=True, max_length=150, return_tensors="pt").to(device)

    labels = tgt["input_ids"].clone()
    labels[labels == tok.pad_token_id] = -100  # ✅ PAD 不算 loss

    model.refiner.train()
    out = model.refiner(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        labels=labels,
    )

    loss = out.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


In [8]:
import random
from rouge_score import rouge_scorer

rouge_scorer_l = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_rouge_l(pred, ref):
    if ref is None or ref.strip() == "":
        return None
    score = rouge_scorer_l.score(ref, pred)
    return score["rougeL"].fmeasure


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Build Dataset  (✅ 重新实例化，确保 N/A 已过滤)
dataset = SPoRCDataset(
    csv_path="sporc_turns_selected_clean.csv",
    summary_jsonl_path="sporc_full_summaries_7b.jsonl",
    device=device,
)

# 2) Build Model
model = TwoStageSummarizer().to(device)

# 3) Add control tokens (run once per new model/tokenizer)
dataset.tokenizer.add_tokens(["<GLOBAL>", "<HOST>", "<GUEST>"])
model.refiner.model.resize_token_embeddings(len(dataset.tokenizer))  # ✅ 注意这里是 model.refiner.model

# 4) Train/Test split：只用 global 有内容的 episode（避免空跑）
episodes_global = [
    ep for ep, s in dataset.summaries.items()
    if ep in dataset.episodes and s.get("global","").strip() != ""
]
# 4) Train/Test split：只用 global 有内容的 episode（避免空跑）
episodes_global = [
    ep for ep, s in dataset.summaries.items()
    if ep in dataset.episodes and s.get("global","").strip() != ""
]

train_episodes = episodes_global[1000:2000]
test_episodes  = episodes_global[2000:2200]


# =======================
# Part A: Train Extractor
# =======================
print("\n===== Train Extractor =====")
for st in ["global", "host", "guest"]:
    loss = train_extractor(
        model,
        dataset,
        train_episodes,
        summary_type=st,
    )
    print(f"[Extractor] {st} average loss: {loss:.4f}")

# =======================
# Part C: Train Refiner (✅ 混合训练，避免 guest 把模型冲成 N/A)
# =======================
print("\n===== Train Refiner (mixed) =====")
refiner_optimizer = torch.optim.AdamW(model.refiner.parameters(), lr=2e-5)

pairs = []
for ep in train_episodes:
    for st in ["global", "host", "guest"]:
        # 只训练有 reference 的
        if dataset.get_summary(ep, st).strip() != "":
            pairs.append((ep, st))

random.shuffle(pairs)

losses = {"global": [], "host": [], "guest": []}
for ep, st in pairs:
    l = train_refiner_on_episode(model, dataset, ep, summary_type=st, optimizer=refiner_optimizer)
    if l is not None:
        losses[st].append(l)

for st in ["global", "host", "guest"]:
    if len(losses[st]) > 0:
        print(f"[Refiner] {st} average loss: {sum(losses[st])/len(losses[st]):.4f}")
    else:
        print(f"[Refiner] {st}: no valid samples")

# =======================
# Part D: Evaluate (ROUGE-L)
# =======================
print("\n===== Evaluate Refiner (ROUGE-L) =====")

rouge_scores = {"global": [], "host": [], "guest": []}

for ep in test_episodes:
    for st in ["global", "host", "guest"]:
        gold = dataset.get_summary(ep, st)
        if gold.strip() == "":
            continue

        pred = generate_summary(model, dataset, ep, summary_type=st)
        score = compute_rouge_l(pred, gold)
        if score is not None:
            rouge_scores[st].append(score)

for st in ["global", "host", "guest"]:
    if len(rouge_scores[st]) > 0:
        avg = sum(rouge_scores[st]) / len(rouge_scores[st])
        print(f"ROUGE-L ({st}): {avg:.4f}")
    else:
        print(f"ROUGE-L ({st}): N/A")

# =======================
# Part E: Show one example
# =======================
def has_role_turn(ep, role_prefix):
    _, _, roles, _ = dataset.encode_episode(ep)
    role_prefix = role_prefix.lower()
    return any(((r or "").strip().lower().startswith(role_prefix)) for r in roles)

# ✅ 选一个：gold guest 非空 + transcript 里确实有 guest turn
ep = next(
    ep for ep in test_episodes
    if dataset.get_summary(ep, "guest").strip() != "" and has_role_turn(ep, "guest")
)

global_summary = generate_summary(model, dataset, ep, summary_type="global")
host_summary   = generate_summary(model, dataset, ep, summary_type="host")
guest_summary  = generate_summary(model, dataset, ep, summary_type="guest")

episode_result = {
    "episode": ep,
    "summaries": {"global": global_summary, "host": host_summary, "guest": guest_summary},
    "conflict": {"label": "NEUTRAL", "score": 0.0, "evidence": []}
}

print("\n--- Example Summaries ---")
print("picked ep:", ep)
print("\n[GLOBAL]\n", global_summary)
print("\n[HOST]\n", host_summary)
print("\n[GUEST]\n", guest_summary)
print("\nQ3:", answer_question("What is the guest view?", episode_result))


global_summary = generate_summary(model, dataset, ep, summary_type="global")
host_summary   = generate_summary(model, dataset, ep, summary_type="host")
guest_summary  = generate_summary(model, dataset, ep, summary_type="guest")

print("\n--- Example Summaries ---")
print("\n[GLOBAL]\n", global_summary)
print("\n[HOST]\n", host_summary)
print("\n[GUEST]\n", guest_summary)


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



===== Train Extractor =====
[Extractor] global average loss: 0.6625
[Extractor] host average loss: 0.6318
[Extractor] guest average loss: 0.7521

===== Train Refiner (mixed) =====
[Refiner] global average loss: 2.6161
[Refiner] host average loss: 2.7814
[Refiner] guest average loss: 3.2759

===== Evaluate Refiner (ROUGE-L) =====
ROUGE-L (global): 0.2331
ROUGE-L (host): 0.2354
ROUGE-L (guest): 0.1834

--- Example Summaries ---
picked ep: https://anchor.fm/s/10858c10/podcast/play/13990521/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fproduction%2F2020-4-19%2F74721813-44100-2-e9b1366301e55.mp3

[GLOBAL]
 The host introduces Bob Gilligan, welcomes guests Kathy Stauber and Mary Lou Kelly, and provides details about the new program 'C-A-L-E-D-2-T-O-P-R-A-'E-R' and encourages listeners to pray for Saint John Paul II on May 18th, 2020. He also mentions upcoming events such as the commemoration of Father Tilka's ordination to the state of Illinois.

[HOST]
 The host introduces the topic of Sai

NameError: name 'answer_question' is not defined

In [9]:
# ======== Cell 8 (REPLACE WHOLE CELL) ========
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ✅ 建议优先用 base（更快更省显存），不行再用 large
CANDIDATES = [
    "MoritzLaurer/DeBERTa-v3-base-mnli",     # 推荐：快、好用
    "microsoft/deberta-large-mnli",          # 官方 MNLI（不是 v3）
    "roberta-large-mnli",                   # 常用 MNLI
    "facebook/bart-large-mnli",             # 常用 MNLI
    "potsawee/deberta-v3-large-mnli",       # v3-large（更慢更大）
]

last_err = None
nli_tokenizer = None
nli_model = None
nli_model_name = None

for name in CANDIDATES:
    try:
        nli_tokenizer = AutoTokenizer.from_pretrained(name)
        nli_model = AutoModelForSequenceClassification.from_pretrained(name).to(device)
        nli_model.eval()
        nli_model_name = name
        print("Loaded NLI model:", name)
        break
    except Exception as e:
        print("Failed:", name, "|", repr(e))
        last_err = e

if nli_model is None:
    raise RuntimeError(f"All NLI model candidates failed. Last error: {last_err}")

# ---------- 自动 label 映射（不同模型顺序可能不一样） ----------
id2label = nli_model.config.id2label
# 统一成 int->str
id2label = {int(k): str(v).upper() for k, v in id2label.items()}

def _norm_label(s: str) -> str:
    s = s.upper()
    if "CONTRAD" in s:
        return "CONTRADICTION"
    if "ENTAIL" in s:
        return "ENTAILMENT"
    if "NEUTRAL" in s:
        return "NEUTRAL"
    return s

LABEL_MAP = {i: _norm_label(lbl) for i, lbl in id2label.items()}

@torch.no_grad()
def nli_predict(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    logits = nli_model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    label_id = int(torch.argmax(probs).item())

    return {
        "label": LABEL_MAP.get(label_id, str(label_id)),
        "score": float(probs[label_id].item()),
        "probs": {LABEL_MAP.get(i, str(i)): float(probs[i].item()) for i in range(probs.numel())},
        "model": nli_model_name,
    }


Failed: MoritzLaurer/DeBERTa-v3-base-mnli | ValueError("Converting from SentencePiece and Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast converters: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loaded NLI model: microsoft/deberta-large-mnli


In [10]:
# =========== Cell 9: Local Conflict Evidence (FIXED) =====
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

sent_encoder = SentenceTransformer("all-MiniLM-L6-v2").to(device)

def split_sentences(text):
    return nltk.sent_tokenize(text)

@torch.no_grad()
def find_conflict_evidence(host_summary, guest_summary, topk=3):
    host_sents = split_sentences(host_summary)
    guest_sents = split_sentences(guest_summary)

    if len(host_sents) == 0 or len(guest_sents) == 0:
        return []

    host_emb = sent_encoder.encode(
        host_sents, convert_to_tensor=True
    )
    guest_emb = sent_encoder.encode(
        guest_sents, convert_to_tensor=True
    )

    sim = util.cos_sim(host_emb, guest_emb)

    pairs = []
    max_pairs = min(
        topk,
        sim.size(0) * sim.size(1)
    )

    # top-k pair 
    sim_copy = sim.clone()
    for _ in range(max_pairs):
        idx = torch.argmax(sim_copy)
        h, g = divmod(idx.item(), sim_copy.size(1))
        pairs.append((host_sents[h], guest_sents[g]))
        sim_copy[h, g] = -1

    evidence = []
    for h_sent, g_sent in pairs:
        nli_res = nli_predict(h_sent, g_sent)
        if nli_res["label"] == "CONTRADICTION":
            evidence.append({
                "host_sentence": h_sent,
                "guest_sentence": g_sent,
                "confidence": nli_res["score"],
            })

    return evidence


In [11]:
# =========== Cell 10: Final Structured Output =====

def build_conflict_output(episode, host_summary, guest_summary):
    overall = nli_predict(host_summary, guest_summary)
    evidence = find_conflict_evidence(
        host_summary, guest_summary
    )

    return {
        "episode": episode,
        "conflict_overall": {
            "label": overall["label"],
            "score": overall["score"],
        },
        "conflict_type": (
            "stance_disagreement"
            if overall["label"] == "CONTRADICTION"
            else "none"
        ),
        "evidence": evidence,
    }


In [12]:
# =============== Cell 11: Q&A =================

def answer_question(question, episode_result):
    q = question.lower()

    # ---- unpack with safety ----
    summaries = episode_result.get("summaries", {})
    conflict  = episode_result.get("conflict", {})

    conflict_label = conflict.get("label", "NEUTRAL")
    evidence = conflict.get("evidence", [])

    # ---- Conflict details (must be BEFORE general conflict) ----
    if "what" in q and "disagree" in q:
        if conflict_label != "CONTRADICTION":
            return "No specific disagreement was identified in this episode."

        if len(evidence) == 0:
            return (
                "A disagreement was detected, but no specific "
                "evidence sentence pair was identified."
            )

        e = evidence[0]
        return (
            "The disagreement is reflected in the following statements:\n"
            f"- Host: {e.get('host_sentence', '')}\n"
            f"- Guest: {e.get('guest_sentence', '')}"
        )

    # ---- Conflict existence ----
    if "disagree" in q or "conflict" in q:
        if conflict_label == "CONTRADICTION":
            return (
                "Yes. A disagreement between the host and the guest "
                "was detected."
            )
        else:
            return (
                "No clear disagreement between the host and the guest "
                "was detected."
            )

    # ---- Role-specific viewpoints ----
    if "host" in q and ("view" in q or "opinion" in q):
        return summaries.get("host", "No host-specific summary available.")

    if "guest" in q and ("view" in q or "opinion" in q):
        return summaries.get("guest", "No guest-specific summary available.")

    # ---- Global summary ----
    if "summary" in q or "about" in q:
        return summaries.get("global", "No global summary available.")

    # ---- Fallback ----
    return "This question is not supported by the current QA system."


In [15]:
def run_episode(model, dataset, episode):
    # 1) summaries
    g = generate_summary(model, dataset, episode, "global")
    h = generate_summary(model, dataset, episode, "host")
    ge = generate_summary(model, dataset, episode, "guest")

    # 2) conflict（如果 host/guest 任意为空，就认为无法判定）
    conflict = {"label": "NEUTRAL", "score": 0.0, "evidence": []}
    if (h or "").strip() != "" and (ge or "").strip() != "":
        conflict_obj = build_conflict_output(episode, h, ge)
        conflict = {
            "label": conflict_obj["conflict_overall"]["label"],
            "score": conflict_obj["conflict_overall"]["score"],
            "evidence": conflict_obj["evidence"],
        }

    return {
        "episode": episode,
        "summaries": {"global": g, "host": h, "guest": ge},
        "conflict": conflict
    }

# quick test
ep = test_episodes[0]
episode_result = run_episode(model, dataset, ep)
episode_result


{'episode': 'https://anchor.fm/s/10858c10/podcast/play/13644979/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fproduction%2F2020-4-12%2F72469431-44100-2-c7b953420201e.mp3',
 'summaries': {'global': 'The podcast episode features Hector Rivera and Jessica Mead discussing their experiences as therapists at Catholic Charities Youth Counseling Programs, focusing on the challenges faced by families during the pandemic. The conversation covers topics such as the importance of communication, coping mechanisms, and coping mechanisms. The hosts also share personal anecdotes about their experiences and advice for families dealing with mental health issues.',
  'host': 'The podcast episode features Hector Rivera and Jessica Mead discussing their experiences as therapists at Catholic Charities Youth Counseling Programs, focusing on the challenges faced by families during the pandemic. The conversation covers topics such as the importance of communication, coping mechanisms, and coping mechanisms. Th

In [14]:
print("Episode:", episode_result["episode"])
print("\nQ1:", answer_question("What is this episode about?", episode_result))
print("\nQ2:", answer_question("What is the host view?", episode_result))
print("\nQ3:", answer_question("What is the guest view?", episode_result))
print("\nQ4:", answer_question("Do they disagree?", episode_result))
print("\nQ5:", answer_question("What do they disagree about?", episode_result))


Episode: https://anchor.fm/s/10858c10/podcast/play/13644979/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fproduction%2F2020-4-12%2F72469431-44100-2-c7b953420201e.mp3

Q1: The podcast episode features Hector Rivera and Jessica Mead discussing their experiences as therapists at Catholic Charities Youth Counseling Programs, focusing on the challenges faced by families during the pandemic. The conversation covers topics such as the importance of communication, coping mechanisms, and coping mechanisms. The hosts also share personal anecdotes about their experiences and advice for families dealing with mental health issues.

Q2: The podcast episode features Hector Rivera and Jessica Mead discussing their experiences as therapists at Catholic Charities Youth Counseling Programs, focusing on the challenges faced by families during the pandemic. The conversation covers topics such as the importance of communication, coping mechanisms, and coping mechanisms. The hosts also share personal anecdot

In [16]:
import json
from tqdm import tqdm

OUTPUT_JSONL = "test_results.jsonl"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for ep in tqdm(test_episodes):
        r = run_episode(model, dataset, ep)
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:", OUTPUT_JSONL)


100%|██████████| 200/200 [06:07<00:00,  1.84s/it]

Saved: test_results.jsonl


In [17]:
import json

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

results = load_jsonl("test_results.jsonl")

has_guest = sum(1 for r in results if (r["summaries"].get("guest","") or "").strip() != "")
has_conflict = sum(1 for r in results if r["conflict"].get("label") == "CONTRADICTION")

print("Total:", len(results))
print("Has guest summary:", has_guest)
print("Detected contradiction:", has_conflict)


Total: 200
Has guest summary: 9
Detected contradiction: 2


In [ ]:
# =====  the result of finding a conflicting sample and running an existing QA test.
import random

# results 已在上面 load_jsonl 读出来了；如果没有，就取消下一行注释重新读
# results = load_jsonl("test_results.jsonl")

contradictions = [
    r for r in results
    if r.get("conflict", {}).get("label") == "CONTRADICTION"
]

print("CONTRADICTION count:", len(contradictions))

if len(contradictions) == 0:
    print("没在 test_results.jsonl 里找到 CONTRADICTION。")
else:
    # 选第一个或随机一个都行
    episode_result = random.choice(contradictions)
    # episode_result = contradictions[0]

    print("\n========== PICKED EPISODE ==========")
    print("Episode:", episode_result["episode"])
    print("Conflict label/score:", episode_result["conflict"].get("label"), episode_result["conflict"].get("score"))

    ev = episode_result["conflict"].get("evidence", [])
    print("\nTop evidence pairs:", len(ev))
    if ev:
        e0 = ev[0]
        print("- Host:", e0.get("host_sentence", ""))
        print("- Guest:", e0.get("guest_sentence", ""))

    print("\n========== QA ==========")
    print("\nQ1:", answer_question("What is this episode about?", episode_result))
    print("\nQ2:", answer_question("What is the host view?", episode_result))
    print("\nQ3:", answer_question("What is the guest view?", episode_result))
    print("\nQ4:", answer_question("Do they disagree?", episode_result))
    print("\nQ5:", answer_question("What do they disagree about?", episode_result))


CONTRADICTION count: 2

========== PICKED EPISODE ==========
Episode: https://anchor.fm/s/10858c10/podcast/play/15226445/https%3A%2F%2Fd3ctxlq1ktw2nl.cloudfront.net%2Fproduction%2F2020-5-15%2F82499051-44100-2-e7312cb0e0121.mp3
Conflict label/score: CONTRADICTION 0.8907063007354736

Top evidence pairs: 1
- Host: Bob Gilligan introduces Pete Newburn, an Ecumenical officer with the Diocese of Joliet, as a speaker at a virtual prayer rally for racial justice.
- Guest: The host, Bishop Tate, leads the discussion, emphasizing the importance of recognizing and addressing systemic racism and the need for systemic change.

========== QA ==========

Q1: Bob Gilligan introduces Pete Newburn, an Ecumenical officer with the Diocese of Joliet, as a speaker at a virtual prayer rally for racial justice. He discusses the significance of the rally, its purpose, and the role of the Catholic Conference in organizing it. The host also encourages listeners to visit the diocese's website for more information